In [ ]:
# --- Step 1: 环境设置 ---
%pip install transformers datasets tokenizers sentencepiece

import torch
import torch.nn as nn
import math
from typing import Tuple

# --- Step 2: 从 model.py 导入所有组件 ---
from model import (
    ModelConfig,
    RMSNorm,
    Attention,
    MLP,
    DecoderLayer,
    Transformer,
    precompute_freqs_cis
)

print("所有模块和依赖导入成功！")

In [ ]:
# ======================================================
# Step 3: 定义用于实验的模型配置
# ======================================================

args = ModelConfig(
    dim=128,              # 模型维度
    n_layers=4,           # Transformer 层数
    n_heads=4,            # 注意力头数
    n_kv_heads=2,         # GQA
    vocab_size=1024,      # 词汇表大小
    max_seq_len=256,      # 最大序列长度
    dropout=0.0,
    flash_attn=False
)

print("模型配置创建成功：")
print(args)

In [ ]:
# ======================================================
# Step 4: 对核心组件进行单元测试
# ======================================================

# --- 测试参数 ---
batch_size = 2
seq_len = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}\n")

# --- 4.1 测试 RMSNorm ---
print("--- 测试 RMSNorm ---")
norm_layer = RMSNorm(args.dim, args.norm_eps).to(device)
test_input = torch.randn(batch_size, seq_len, args.dim).to(device)
output = norm_layer(test_input)
assert output.shape == test_input.shape
print("RMSNorm 测试通过！\n")

# --- 4.2 测试 Attention ---
print("--- 测试 Attention ---")
attn_layer = Attention(args).to(device)
freqs_cos, freqs_sin = precompute_freqs_cis(args.dim // args.n_heads, seq_len)
freqs_cos, freqs_sin = freqs_cos.to(device), freqs_sin.to(device)
output = attn_layer(test_input, freqs_cos, freqs_sin)
assert output.shape == test_input.shape
print("Attention 测试通过！\n")

# --- 4.3 测试 MLP ---
print("--- 测试 MLP ---")
mlp_layer = MLP(args.dim, args.hidden_dim, args.multiple_of, args.dropout).to(device)
output = mlp_layer(test_input)
assert output.shape == test_input.shape
print("MLP 测试通过！\n")

# --- 4.4 测试 DecoderLayer ---
print("--- 测试 DecoderLayer ---")
decoder_layer = DecoderLayer(layer_id=0, args=args).to(device)
output = decoder_layer(test_input, freqs_cos, freqs_sin)
assert output.shape == test_input.shape
print("DecoderLayer 测试通过！")

In [ ]:
# ======================================================
# Step 5: 端到端测试
# ======================================================

# --- 5.1 实例化完整模型 ---
print("--- 实例化完整的 Transformer 模型 ---")
model = Transformer(args).to(device)
print("模型结构:")
print(model)
print(f"\n模型总参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

# --- 5.2 测试前向传播 (模拟训练) ---
print("\n--- 测试前向传播 (模拟训练) ---")
# 模拟输入 token ids 和 targets
input_ids = torch.randint(0, args.vocab_size, (batch_size, seq_len)).to(device)
targets = torch.randint(0, args.vocab_size, (batch_size, seq_len)).to(device)

# 前向传播
output = model(tokens=input_ids, targets=targets)
logits = output.logits
loss = output.last_loss

print(f"Logits 的形状: {logits.shape}")
print(f"Loss 的形状: {loss.shape if loss is not None else 'None'}")
# 预期 Logits 形状: [batch_size, seq_len, vocab_size]
assert logits.shape == (batch_size, seq_len, args.vocab_size)
print("前向传播 (训练模式) 测试通过！\n")

# --- 5.3 测试生成 (模拟推理) ---
print("--- 测试生成 (模拟推理) ---")
# 模拟一个起始 prompt
prompt_ids = torch.randint(0, args.vocab_size, (1, 5)).to(device) # batch_size=1, 长度为5

# 调用 generate 函数
# temperature > 0 用于随机采样, top_k 限制采样范围
generated_ids = model.generate(prompt_ids, max_new_tokens=10, temperature=0.8, top_k=20)

print(f"输入的 Prompt IDs: {prompt_ids}")
print(f"生成的 Token IDs: {generated_ids}")
print(f"生成序列的总长度: {generated_ids.shape[1]}")
# 预期生成长度是 10
assert generated_ids.shape[1] == 10
print("生成 (推理模式) 测试通过！")